# 01 · Fuentes — Gafas GRADUADAS

**Proyecto ML — Predicción de precios de gafas (caso óptica).**

Fuente: **Lentiamo España** — solo categoría **gafas graduadas**.



## Salida
`data/raw/lentiamo_graduadas.csv`

## 0. Instalación de dependencias

In [1]:
%pip install -q requests beautifulsoup4 lxml pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\juan_\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1. Imports y configuración

In [2]:
import csv
import json
import logging
import random
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
CSV_OUT = RAW / 'lentiamo_graduadas.csv'
print('Guardaré los datos en:', CSV_OUT)

BASE_URL = 'https://www.lentiamo.es'

# SOLO graduadas
CATEGORIAS = {
    'graduadas': 'https://www.lentiamo.es/gafas-graduadas.html',
}

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate',
}

DELAY_RANGE = (0.8, 2.0)
REQUEST_TIMEOUT = 20
MAX_REINTENTOS = 3

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
log = logging.getLogger('lentiamo-graduadas')


def construir_url(path_o_url, query=''):
    base = path_o_url if path_o_url.startswith('http') else BASE_URL + path_o_url
    if query:
        sep = '&' if '?' in base else '?'
        base = base + sep + query
    return base

Guardaré los datos en: c:\Users\juan_\Desktop\THE-BRIDGE\RepositorioJuanAMM\Proyecto ML\data\raw\lentiamo_graduadas.csv


## 2. Modelo de datos + descarga

Sin `valoracion` ni `num_resenas` (los widgets de Feedaty se cargan por JS, no merece la pena).

In [3]:
@dataclass
class Producto:
    url: str
    marca: Optional[str] = None
    modelo: Optional[str] = None
    tipo: Optional[str] = None
    genero: Optional[str] = None
    material_montura: Optional[str] = None
    forma: Optional[str] = None
    tipo_montura: Optional[str] = None
    color: Optional[str] = None
    talla: Optional[str] = None
    ancho_lente: Optional[float] = None
    ancho_puente: Optional[float] = None
    largo_varilla: Optional[float] = None
    calibre_total: Optional[float] = None
    peso: Optional[float] = None
    polarizadas: Optional[bool] = None
    precio: Optional[float] = None
    raw_specs: dict = field(default_factory=dict)


def get_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def fetch(session, url):
    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            r = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code == 200:
                return r.text
            if r.status_code in (403, 429):
                log.warning('[%s] %s', r.status_code, url)
                time.sleep(5 * intento)
                continue
            log.warning('[%s] %s', r.status_code, url)
            return None
        except requests.RequestException as e:
            log.warning('Error en %s: %s (intento %d)', url, e, intento)
            time.sleep(2 * intento)
    return None


def polite_sleep():
    time.sleep(random.uniform(*DELAY_RANGE))

## 3. Filtro de URLs y descubrimiento por categoría

In [4]:
PAGINAS_NO_PRODUCTO = {
    'politica-de-cookies', 'cesta', 'olvidar-datos-de-acceso',
    'cheque-regalo', 'productos-extra', 'formas-de-envio',
    'liquidos', 'gotas-oculares',
    'lentillas-baratas', 'lentillas-diarias', 'lentillas-quincenales',
    'lentillas-mensuales', 'lentillas-trimestrales', 'lentillas-uso-prolongado',
    'lentillas-esfericas',
    'lentes-de-contacto-toricas-para-astigmatismo',
    'lentes-bifocales-y-multifocales', 'lentes-de-hidrogel-de-silicona',
    'lentes-de-colores', 'lentes-de-paquetes',
    'acuvue', 'lentes-biofinity', 'dailies', 'lentes-air-optix', 'lenjoy',
    'soflens', 'lentes-purevision', 'lentes-proclear', 'clariti',
    'precision1', 'lentes-total30',
    'gafas-graduadas', 'gafas-de-sol', 'gafas-luz-azul', 'gafas-de-lectura',
    'guia-de-gafas', 'como-comprar-gafas-graduadas-online',
    'login', 'sobre-nosotros', 'contacto', 'devoluciones', 'condiciones',
}

PREFIJOS_LENTILLAS = (
    'lentillas-', 'lentes-de-', 'lente-',
    'lenjoy-', 'acuvue-', 'biofinity-', 'air-optix-',
    'soflens-', 'clariti-', 'dailies-', 'precision1-',
    'purevision-', 'proclear-', 'total30-', 'oasys-',
    'liquido-', 'gotas-',
)


def es_url_producto(href):
    if not href.endswith('.html') or not href.startswith('/'):
        return False
    path = href.split('?')[0]
    segmentos = [s for s in path.split('/') if s]
    if len(segmentos) != 1:
        return False
    slug = segmentos[0].replace('.html', '')
    if slug in PAGINAS_NO_PRODUCTO:
        return False
    if slug.startswith(PREFIJOS_LENTILLAS):
        return False
    tokens = slug.split('-')
    if len(tokens) < 3:
        return False
    if not any(re.search(r'\d', t) for t in tokens):
        return False
    return True


def urls_desde_categoria(session, path_categoria, max_paginas=80):
    urls = []
    for pagina in range(1, max_paginas + 1):
        url = construir_url(path_categoria, query=f'page={pagina}')
        html = fetch(session, url)
        polite_sleep()
        if not html:
            break
        soup = BeautifulSoup(html, 'html.parser')
        page_urls = set()
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.startswith(BASE_URL):
                href = href[len(BASE_URL):]
            if href.startswith('http'):
                continue
            if es_url_producto(href):
                page_urls.add(BASE_URL + href.split('?')[0])
        if not page_urls:
            log.info('Sin productos en pág %d → fin paginación', pagina)
            break
        antes = len(set(urls))
        urls.extend(page_urls)
        ahora = len(set(urls))
        log.info('Cat pág %d → %d enlaces (acum únicos: %d)', pagina, len(page_urls), ahora)
        if ahora == antes:
            break
    return list(dict.fromkeys(urls))

## 4. Parser de la tabla `vc-table-dotted`

In [5]:
def _to_float(val):
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).replace(',', '.').strip()
    m = re.search(r'-?\d+(?:\.\d+)?', s)
    return float(m.group()) if m else None


def _bool_es(s):
    s = s.lower().strip()
    if s in {'sí', 'si', 'yes', 'true', '1'}:
        return True
    if s in {'no', 'false', '0'}:
        return False
    return None


def parse_tabla_lentiamo(soup):
    """Extrae specs de tabla con <tr><th>clave:</th><td>valor</td></tr>."""
    specs = {}
    nombre_completo = None
    for tr in soup.find_all('tr'):
        th = tr.find('th')
        td = tr.find('td')
        if not th or not td:
            continue
        try:
            if int(th.get('colspan', 1)) >= 2:
                continue
        except (ValueError, TypeError):
            pass
        k = th.get_text(' ', strip=True).replace('\xa0', ' ').rstrip(':').strip().lower()
        v = td.get_text(' ', strip=True).replace('\xa0', ' ').strip()
        if k and v and len(k) < 60 and len(v) < 200:
            specs[k] = v
        if nombre_completo is None and td.get('data-thname'):
            nombre_completo = td['data-thname'].strip()
    return specs, nombre_completo


MAPEO_SPECS = {
    'marca': 'marca',
    'modelo': 'modelo',
    'código': 'modelo',
    'codigo': 'modelo',
    'género': 'genero',
    'genero': 'genero',
    'sexo': 'genero',
    'categoría': 'tipo',
    'categoria': 'tipo',
    'material de la montura': 'material_montura',
    'material': 'material_montura',
    'forma de la montura': 'forma',
    'forma': 'forma',
    'tipo de montura': 'tipo_montura',
    'color de la montura': 'color',
    'color': 'color',
    'tallas': 'talla',
    'talla': 'talla',
    'ancho del cristal': 'ancho_lente',
    'puente': 'ancho_puente',
    'largo de las patillas': 'largo_varilla',
    'patillas': 'largo_varilla',
    'calibre total de las gafas': 'calibre_total',
    'calibre total': 'calibre_total',
    'peso': 'peso',
    'polarizada': 'polarizadas',
    'polarizadas': 'polarizadas',
}
CAMPOS_NUMERICOS = {'ancho_lente', 'ancho_puente', 'largo_varilla', 'calibre_total', 'peso'}


def precio_desde_html(html, soup):
    m = soup.find('meta', attrs={'property': 'product:price:amount'})
    if m and m.get('content'):
        v = _to_float(m['content'])
        if v: return v
    m = soup.find('meta', attrs={'property': 'og:price:amount'})
    if m and m.get('content'):
        v = _to_float(m['content'])
        if v: return v
    for el in soup.find_all(attrs={'itemprop': 'price'}):
        v = _to_float(el.get('content') or el.get_text(' ', strip=True))
        if v: return v
    m = re.search(r'"price"\s*:\s*"?(\d+[\.,]\d{2})"?', html)
    if m: return _to_float(m.group(1))
    m = re.search(r'(\d{1,4}[,\.]\d{2})\s*€', html)
    if m: return _to_float(m.group(1))
    return None


def parse_producto(html, url, tipo_default=None):
    soup = BeautifulSoup(html, 'html.parser')
    p = Producto(url=url, tipo=tipo_default)
    specs, nombre = parse_tabla_lentiamo(soup)
    p.raw_specs = specs
    for k_es, attr in MAPEO_SPECS.items():
        if k_es in specs and not getattr(p, attr):
            valor = specs[k_es]
            if attr in CAMPOS_NUMERICOS:
                setattr(p, attr, _to_float(valor))
            elif attr == 'polarizadas':
                setattr(p, attr, _bool_es(valor))
            else:
                setattr(p, attr, re.sub(r'\s+', ' ', valor).strip())
    if nombre and not p.modelo:
        p.modelo = nombre
    elif not p.modelo and soup.title and soup.title.string:
        p.modelo = soup.title.string.strip()
    if p.tipo:
        t = p.tipo.lower()
        if 'sol' in t:
            p.tipo = 'sol'
        elif 'graduad' in t:
            p.tipo = 'graduadas'
    if p.precio is None:
        p.precio = precio_desde_html(html, soup)
    return p if (p.precio is not None or p.marca) else None

## 5. Guardado y orquestador

In [6]:
CSV_FIELDS = [
    'url', 'marca', 'modelo', 'tipo', 'genero', 'material_montura',
    'forma', 'tipo_montura', 'color', 'talla',
    'ancho_lente', 'ancho_puente', 'largo_varilla', 'calibre_total', 'peso',
    'polarizadas', 'precio',
]


def cargar_urls_existentes(csv_path):
    if not csv_path.exists():
        return set()
    with csv_path.open(encoding='utf-8') as f:
        return {row['url'] for row in csv.DictReader(f) if row.get('url')}


def escribir_csv(csv_path, productos, modo_append):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    nuevo = not csv_path.exists()
    with csv_path.open('a' if modo_append else 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if nuevo or not modo_append:
            writer.writeheader()
        for p in productos:
            writer.writerow({k: getattr(p, k) for k in CSV_FIELDS})


def scrape(max_productos=3000, salida=CSV_OUT):
    session = get_session()
    urls_por_tipo = {}
    for tipo, path in CATEGORIAS.items():
        log.info('Buscando URLs en categoría %s', tipo)
        urls_por_tipo[tipo] = urls_desde_categoria(session, path)
    todas, vistas, deduped = [], set(), []
    for tipo, urls in urls_por_tipo.items():
        for u in urls:
            todas.append((u, tipo))
    for u, tipo in todas:
        if u not in vistas:
            vistas.add(u)
            deduped.append((u, tipo))
    log.info('Total URLs únicas: %d', len(deduped))
    ya = cargar_urls_existentes(salida)
    pendientes = [(u, t) for u, t in deduped if u not in ya]
    log.info('Pendientes: %d (ya teníamos %d)', len(pendientes), len(ya))
    buffer, procesados = [], 0
    for url, tipo in pendientes:
        if procesados >= max_productos:
            break
        html = fetch(session, url)
        polite_sleep()
        if not html:
            continue
        producto = parse_producto(html, url, tipo_default=tipo)
        if producto is None:
            continue
        buffer.append(producto)
        procesados += 1
        if procesados % 25 == 0:
            log.info('Progreso: %d productos extraídos', procesados)
        if len(buffer) >= 50:
            escribir_csv(salida, buffer, modo_append=True)
            buffer.clear()
    if buffer:
        escribir_csv(salida, buffer, modo_append=True)
    log.info('Hecho. Productos nuevos: %d → %s', procesados, salida)
    return procesados

---
## 6. Test del parser con UNA URL real

In [7]:
URL_TEST = 'https://www.lentiamo.es/bottega-veneta-bv1449o-001-52.html'

session = get_session()
html = fetch(session, URL_TEST)
polite_sleep()

if html:
    print(f'✅ HTML descargado: {len(html):,} bytes\n')
    producto = parse_producto(html, URL_TEST, tipo_default='graduadas')
    if producto:
        print('Producto extraído:')
        print('-' * 60)
        for campo, valor in vars(producto).items():
            if campo == 'raw_specs':
                continue
            check = '✓' if valor not in (None, '') else '✗'
            print(f'  [{check}] {campo:<20} {valor!r}')
        rellenos = sum(1 for k, v in vars(producto).items()
                       if k != 'raw_specs' and v not in (None, ''))
        total = len(vars(producto)) - 1
        print(f'\nCobertura: {rellenos}/{total}')
    else:
        print('⚠ Parser devolvió None.')
else:
    print('❌ No se descargó')

✅ HTML descargado: 291,864 bytes

Producto extraído:
------------------------------------------------------------
  [✓] url                  'https://www.lentiamo.es/bottega-veneta-bv1449o-001-52.html'
  [✓] marca                'Bottega Veneta'
  [✓] modelo               'BV1449O 001 52'
  [✓] tipo                 'graduadas'
  [✓] genero               'Mujer'
  [✓] material_montura     'Pasta'
  [✓] forma                'Rectangulares'
  [✓] tipo_montura         'Montura completa'
  [✓] color                'Negro'
  [✓] talla                'S'
  [✓] ancho_lente          52.0
  [✓] ancho_puente         16.0
  [✓] largo_varilla        140.0
  [✓] calibre_total        129.0
  [✓] peso                 160.0
  [✗] polarizadas          None
  [✓] precio               289.9

Cobertura: 16/17


---
## 7. Scraping completo de gafas GRADUADAS

Tarda ~30-90 minutos según volumen. Va guardando cada 50 productos.

In [8]:
scrape(max_productos=3000, salida=CSV_OUT)

11:56:21 [INFO] Buscando URLs en categoría graduadas
11:56:24 [INFO] Cat pág 1 → 42 enlaces (acum únicos: 42)
11:56:26 [INFO] Cat pág 2 → 49 enlaces (acum únicos: 87)
11:56:28 [INFO] Cat pág 3 → 50 enlaces (acum únicos: 133)
11:56:30 [INFO] Cat pág 4 → 52 enlaces (acum únicos: 181)
11:56:32 [INFO] Cat pág 5 → 50 enlaces (acum únicos: 227)
11:56:34 [INFO] Cat pág 6 → 52 enlaces (acum únicos: 275)
11:56:37 [INFO] Cat pág 7 → 52 enlaces (acum únicos: 323)
11:56:39 [INFO] Cat pág 8 → 52 enlaces (acum únicos: 371)
11:56:40 [INFO] Cat pág 9 → 51 enlaces (acum únicos: 418)
11:56:43 [INFO] Cat pág 10 → 52 enlaces (acum únicos: 466)
11:56:45 [INFO] Cat pág 11 → 51 enlaces (acum únicos: 513)
11:56:47 [INFO] Cat pág 12 → 52 enlaces (acum únicos: 561)
11:56:49 [INFO] Cat pág 13 → 52 enlaces (acum únicos: 609)
11:56:51 [INFO] Cat pág 14 → 52 enlaces (acum únicos: 657)
11:56:54 [INFO] Cat pág 15 → 52 enlaces (acum únicos: 705)
11:56:57 [INFO] Cat pág 16 → 52 enlaces (acum únicos: 753)
11:56:59 [INFO

2875

## 8. Inspección del CSV

In [9]:
if CSV_OUT.exists():
    df = pd.read_csv(CSV_OUT)
    print('Filas:', len(df))
    display(df.head())
    print('\n% cobertura por columna:')
    print((df.notna().mean() * 100).round(1).sort_values(ascending=False))
    print('\nDistribución de precio:')
    display(df['precio'].describe())
    print('\nTop 10 marcas:')
    display(df['marca'].value_counts().head(10))
else:
    print('Aún no hay CSV. Ejecuta antes la sección 7.')

Filas: 2875


,url,marca,modelo,tipo,genero,material_montura,forma,tipo_montura,color,talla,ancho_lente,ancho_puente,largo_varilla,calibre_total,peso,polarizadas,precio
0,https://www.lentiamo.es/ray-ban-0rx5698-8109-5...,Ray-Ban,0RX5698 8109 56,graduadas,Unisex,Acetato,Aviador,Montura completa,Marrón,M,56.0,14.0,145.0,133.0,190.0,NaN,112.9
1,https://www.lentiamo.es/saint-laurent-sl-574-0...,Saint Laurent,SL 574 001 52,graduadas,Unisex,Pasta,Rectangulares,Montura completa,Negro,M,52.0,21.0,145.0,139.0,155.0,NaN,249.9
2,https://www.lentiamo.es/saint-laurent-sl-m138-...,Saint Laurent,SL M138 001 55,graduadas,Mujer,Pasta,Cat Eye,Montura completa,Negro,M,55.0,18.0,145.0,138.0,160.0,NaN,249.9
3,https://www.lentiamo.es/oliver-peoples-o-malle...,Oliver Peoples,0OV5183 1003 45,graduadas,Hombre,Acetato,Redondas,Montura completa,Marrón,S,45.0,22.0,145.0,122.0,230.0,NaN,299.9
4,https://www.lentiamo.es/tommy-hilfiger-th-2268...,Tommy Hilfiger,TH 2268/C 807 19 51,graduadas,Unisex,Acetato,Cuadradas,Montura completa,Negro,M,51.0,19.0,150.0,133.0,170.0,NaN,149.9



% cobertura por columna:
url                 100.0
modelo              100.0
tipo                100.0
precio              100.0
marca                99.9
material_montura     99.9
forma                99.9
tipo_montura         99.9
genero               99.9
peso                 99.9
color                99.7
ancho_lente          94.6
ancho_puente         94.6
calibre_total        94.6
largo_varilla        94.6
talla                94.5
polarizadas           0.0
dtype: float64

Distribución de precio:


count    2874.000000
mean      137.054196
std        71.780994
min         3.890000
25%        85.140000
50%       113.900000
75%       179.900000
max       479.900000
Name: precio, dtype: float64


Top 10 marcas:


marca
Ray-Ban              357
Gucci                162
Centrostyle S.p.A    153
Esprit               145
Tom Ford             124
Oakley               119
Vogue                116
Nano Vista           102
Germano Gambini       92
Saint Laurent         91
Name: count, dtype: int64